BiLSTM

In [ ]:
!pip install gensim
!pip install tensorflow
!pip install transformers
from google.colab import files
uploaded=files.upload()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 33.1 MB/s eta 0:00:00


Saving test.csv to test.csv


In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Embedding, Conv1D,LSTM
from tensorflow.keras.layers import MaxPooling1D
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dense,Flatten, Input, Dropout
from sklearn.model_selection import train_test_split
from gensim.parsing.preprocessing import preprocess_string
from nltk import sent_tokenize

In [ ]:
df = pd.read_csv('data_mr.csv').iloc[5000:20000]

In [ ]:
df.drop_duplicates(inplace=True)


In [ ]:
df

,text,label
5000,When I typed Savage Intruder into the IMDb's s...,0
5001,Every once in awhile I'll remember that I've a...,0
5002,"From the brilliant mind that brought us ""The E...",0
5003,Man this thing bites! I am sorry I ever sat do...,0
5004,When I first heared that there was going to be...,0
...,...,...
19995,Very violent and nicely filmed movie. Paulina ...,1
19996,"Is Thursday an original film? Heck no, but it ...",1
19997,I loved this film! Fantastically original and ...,1
19998,This movie was excellent from start-to-finish....,1


In [ ]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)        # remove HTML tags
    text = re.sub(r'[^a-z\s]', '', text)    # keep only letters
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['text'] = df['text'].apply(clean_text)


In [ ]:
df

,text,label
5000,when i typed savage intruder into the imdbs se...,0
5001,every once in awhile ill remember that ive act...,0
5002,from the brilliant mind that brought us the ex...,0
5003,man this thing bites i am sorry i ever sat dow...,0
5004,when i first heared that there was going to be...,0
...,...,...
19995,very violent and nicely filmed movie paulina i...,1
19996,is thursday an original film heck no but it is...,1
19997,i loved this film fantastically original and d...,1
19998,this movie was excellent from starttofinish i ...,1


In [ ]:
y=df['label']

In [ ]:
X=df['text']

In [ ]:
X_train, X_test, y_train, y_test=train_test_split(X,y,test_size=0.2,random_state=7)

In [ ]:
from transformers import BertTokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
texts=X_train.tolist()
encodings = tokenizer(
    texts,
    truncation=True,
    padding='max_length',
    max_length=256,
    return_tensors='np'
)
X_train_ids = encodings['input_ids']

X_test_ids = tokenizer(
    X_test.tolist(),
    truncation=True,
    padding='max_length',
    max_length=256,
    return_tensors='np'
)['input_ids']

In [ ]:
'''model=Sequential()
model.add(Embedding(input_dim=max_words,output_dim=128,input_length=max_len))
model.add(LSTM(128,dropout=0.2,return_sequences=True))
model.add(LSTM(64,dropout=0.2))
model.add(Dense(32, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(16, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(8, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(1, activation='sigmoid'))
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)'''

"model=Sequential()\nmodel.add(Embedding(input_dim=max_words,output_dim=128,input_length=max_len))\nmodel.add(LSTM(128,dropout=0.2,return_sequences=True))\nmodel.add(LSTM(64,dropout=0.2))\nmodel.add(Dense(32, activation='relu'))\nmodel.add(Dropout(0.2))\nmodel.add(Dense(16, activation='relu'))\nmodel.add(Dropout(0.2))\nmodel.add(Dense(8, activation='relu'))\nmodel.add(Dropout(0.2))\nmodel.add(Dense(1, activation='sigmoid'))\nmodel.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])\nearly_stop = EarlyStopping(\n    monitor='val_loss',\n    patience=5,\n    restore_best_weights=True\n)"

In [ ]:
model = Sequential([
    Embedding(
        input_dim=tokenizer.vocab_size,
        output_dim=128,
        input_length=256
    ),
    tf.keras.layers.Bidirectional(
        LSTM(64, dropout=0.3)
    ),
    Dense(32, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

from tensorflow.keras.optimizers import Adam

model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)
model.fit(
    X_train_ids,
    y_train,
    epochs=30,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop]
)

Epoch 1/30
299/299 ━━━━━━━━━━━━━━━━━━━━ 11s 28ms/step - accuracy: 0.5081 - loss: 0.6929 - val_accuracy: 0.5075 - val_loss: 0.6925
Epoch 2/30
299/299 ━━━━━━━━━━━━━━━━━━━━ 8s 26ms/step - accuracy: 0.6650 - loss: 0.6142 - val_accuracy: 0.8411 - val_loss: 0.3994
Epoch 3/30
299/299 ━━━━━━━━━━━━━━━━━━━━ 7s 24ms/step - accuracy: 0.8815 - loss: 0.3189 - val_accuracy: 0.8691 - val_loss: 0.3097
Epoch 4/30
299/299 ━━━━━━━━━━━━━━━━━━━━ 8s 26ms/step - accuracy: 0.9372 - loss: 0.1938 - val_accuracy: 0.8850 - val_loss: 0.2973
Epoch 5/30
299/299 ━━━━━━━━━━━━━━━━━━━━ 7s 24ms/step - accuracy: 0.9572 - loss: 0.1431 - val_accuracy: 0.8704 - val_loss: 0.3602
Epoch 6/30
299/299 ━━━━━━━━━━━━━━━━━━━━ 8s 26ms/step - accuracy: 0.9720 - loss: 0.0960 - val_accuracy: 0.8783 - val_loss: 0.4209
Epoch 7/30
299/299 ━━━━━━━━━━━━━━━━━━━━ 8s 26ms/step - accuracy: 0.9803 - loss: 0.0695 - val_accuracy: 0.8813 - val_loss: 0.3734
Epoch 8/30
299/299 ━━━━━━━━━━━━━━━━━━━━ 7s 23ms/step - accuracy: 0.9863 - loss: 0.0514 - val_acc

In [ ]:
model.evaluate(X_test_ids,y_test)

94/94 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.8913 - loss: 0.2739


[0.27394023537635803, 0.8912679553031921]

In [ ]:
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix

y_pred = (model.predict(X_test_ids) > 0.5).astype(int)
print(f1_score(y_test, y_pred))
print(accuracy_score(y_test, y_pred))


94/94 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step
0.8926329699372316
0.8912679826028772
[[1313  154]
 [ 171 1351]]
